## Data Understanding

### 1. Persiapan

#### 1.1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from config import (
  RAW_REGENCIES_CSV,
  RAW_PROVINCES_CSV,
  NUMERIC_COLUMNS
)

#### 1.2. Persiapan Data

In [ ]:
df_prov = pd.read_csv(RAW_PROVINCES_CSV)
df_reg = pd.read_csv(RAW_REGENCIES_CSV)

### 2. Statistik Deskriptif

In [ ]:
df_reg[NUMERIC_COLUMNS].describe(percentiles=[0.5]).T

In [ ]:
print(f"Total Baris : {len(df_reg)}")
print(f"Total Kolom : {len(df_reg.columns)}")

### 3. Eksplorasi Data

In [ ]:
top_provinces_limit = 10
top_regencies_limit = 10

#### 3.1. Heatmap Korelasi Pearson

In [ ]:
corr_matrix = df_reg[NUMERIC_COLUMNS].corr().round(2)
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriks Korelasi Pearson Indikator KDMP')
plt.tight_layout()
plt.show()

#### 3.2. Barplot Top Provinsi Jumlah Koperasi

In [ ]:
top_prov = df_prov.sort_values(by='total_koperasi', ascending=True).tail(top_provinces_limit)
plt.figure(figsize=(10, 5))
sns.barplot(x='total_koperasi', y='province_name', data=top_prov, hue='province_name', legend=False, palette='Blues_r')
plt.title(f'{top_provinces_limit} Provinsi dengan Jumlah Koperasi Terbanyak di Indonesia')
plt.tight_layout()
plt.show()

#### 3.3. Barplot Top Kabupaten/Kota Nilai Transaksi

In [ ]:
top_reg = df_reg.sort_values(by='nilai_transaksi', ascending=True).tail(top_regencies_limit).copy()
top_reg['nilai_juta'] = top_reg['nilai_transaksi'] / 1e6
plt.figure(figsize=(10, 5))
sns.barplot(x='nilai_juta', y='regency_name', data=top_reg, hue='regency_name', legend=False, palette='viridis')
plt.title(f'{top_regencies_limit} Kabupaten/Kota dengan Nilai Transaksi Tertinggi (Juta Rp)')
plt.tight_layout()
plt.show()

#### 3.4. Distribusi Fitur

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
sns.histplot(df_reg['total_koperasi'], kde=True, ax=axes[0, 0], color='skyblue').set_title('Distribusi Total Koperasi')
sns.histplot(np.log1p(df_reg['simpanan_wajib']), kde=True, ax=axes[0, 1], color='salmon').set_title('Distribusi Log Simpanan Wajib')
sns.histplot(np.log1p(df_reg['volume_transaksi']), kde=True, ax=axes[1, 0], color='lightgreen').set_title('Distribusi Log Volume Transaksi')
sns.histplot(np.log1p(df_reg['nilai_transaksi']), kde=True, ax=axes[1, 1], color='plum').set_title('Distribusi Log Nilai Transaksi')
plt.tight_layout()
plt.show()

#### 3.5. Metrik Lainnya

In [ ]:
total_koperasi = int(df_prov['total_koperasi'].sum())
nib_sum = int(df_prov['koperasi_nib'].sum())
npwp_sum = int(df_prov['koperasi_npwp'].sum())
rat_sum = int(df_prov['koperasi_rat'].sum())
pct_nib = round(nib_sum / total_koperasi * 100, 2)
pct_npwp = round(npwp_sum / total_koperasi * 100, 2)
pct_rat = round(rat_sum / total_koperasi * 100, 2)
simpanan_pokok = float(df_prov['simpanan_pokok'].sum())
simpanan_wajib = float(df_prov['simpanan_wajib'].sum())
nilai_transaksi = float(df_prov['nilai_transaksi'].sum())

print(f"Total Kabupaten/Kota       : {len(df_reg)}")
print(f"Total Koperasi Terdata     : {total_koperasi:,} unit")
print(f"Koperasi Memiliki NIB      : {nib_sum:,} ({pct_nib}%)")
print(f"Koperasi Memiliki NPWP     : {npwp_sum:,} ({pct_npwp}%)")
print(f"Koperasi Telah RAT         : {rat_sum:,} ({pct_rat}%)")
print(f"Akumulasi Simpanan Pokok   : Rp {simpanan_pokok:,.2f}")
print(f"Akumulasi Simpanan Wajib   : Rp {simpanan_wajib:,.2f}")
print(f"Total Nilai Transaksi      : Rp {nilai_transaksi:,.2f}")

### 4. Verifikasi Kualitas Data

#### 4.1. Mengecek Nilai Hilang

In [ ]:
df_reg.isnull().sum()

#### 4.2. Mengecek Outlier

In [ ]:
IGNORED_METADATA_COLUMNS = [
  'cluster_label',
  'regency_name',
  'province_name',
  'no',
  'regency_no',
  'province_id',
  'Province_ID',
  'No',
  'Kabupaten/Kota',
  'latitude',
  'longitude'
]

# Pilih & bersihkan kolom numerik sekaligus
cols = [c for c in df_reg.columns if c not in IGNORED_METADATA_COLUMNS]
df_num = df_reg[cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# 2. Hitung statistik deskriptif dan batas IQR secara vectorized
q1 = df_num.quantile(0.25)
q3 = df_num.quantile(0.75)
iqr = q3 - q1
lower = (q1 - 1.5 * iqr).clip(lower=0)
upper = q3 + 1.5 * iqr

# 3. Hitung jumlah & persentase outlier
out_cnt = ((df_num < lower) | (df_num > upper)).sum()
out_pct = (out_cnt / len(df_num) * 100).round(2)
skew_val = df_num.skew().round(2)

# Buat DataFrame
pd.DataFrame({
  'Nama Fitur': cols,
  'Skewness': skew_val.values,
  'Batas Bawah': lower.round(2).values,
  'Batas Atas': upper.round(2).values,
  'Outliers': out_cnt.values,
  'Persentase (%)': out_pct.values
})